# Taller de Feature Engineering: Seleccion y creacion de variables

**Bootcamp:** Python - PY4
**Modulo:** Machine Learning Avanzado y Feature Engineering

## De donde venimos

En la sesion 2 construimos 3 tecnicas de feature engineering (Winsorization, Target Encoding, variables ciclicas) y vimos que, aplicadas a una Regresion Lineal, el RMSE bajo de 136 a 100 aproximadamente.

## De que trata este material

Hoy vamos a cubrir dos temas que la teoria de la semana menciona pero que todavia no practicamos:

**Parte 1: Creacion de variables de interaccion.** Hasta ahora creamos variables una por una. Hoy vamos a combinar variables entre si (por ejemplo, temperatura x dia laboral) para capturar efectos que ninguna de las dos variables explica por separado.

**Parte 2: Seleccion de caracteristicas.** Cuantas mas variables creamos, mas crece el riesgo de sumar ruido o redundancia. Vamos a ver 3 familias de metodos para decidir que variables realmente vale la pena conservar:

- **Metodos de Filtro**: estadistica pura, comparan cada variable contra el target por separado (correlacion de Pearson, Mutual Information).
- **Metodos Wrapper (envolventes)**: entrenan el modelo una y otra vez, eliminando la variable menos util en cada ronda (Recursive Feature Elimination).
- **Metodos Intrinsecos (Embedded)**: el propio algoritmo de entrenamiento fuerza a cero los coeficientes de las variables menos importantes (Lasso).

Vamos a seguir usando el dataset de Bike Sharing y una Regresion Lineal, igual que en la sesion 2.


## Paso 1: Configuracion y reconstruccion rapida del trabajo previo

Recargamos el dataset y reconstruimos el mismo pipeline de feature engineering de la sesion 2 (Winsorization, Target Encoding, variables ciclicas), para partir del mismo punto.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.feature_selection import mutual_info_regression, RFE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

RANDOM_STATE = 42
sns.set_style("whitegrid")

URL = "https://raw.githubusercontent.com/Camille-Le-Roy/data_analytics_resources/main/Module_02_machine_learning_and_feature_engineering/datasets/bikeshare_Washington.csv"
df = pd.read_csv(URL)

features_base = ["season", "yr", "mnth", "hr", "holiday", "weekday",
                  "workingday", "weathersit", "temp", "atemp", "hum", "windspeed"]

X = df[features_base].copy()
y = df["cnt"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print("Entrenamiento:", X_train.shape, " Prueba:", X_test.shape)


Entrenamiento: (14016, 12)  Prueba: (3504, 12)


In [2]:
def calcular_limites(serie, percentil_bajo=5, percentil_alto=95):
    limite_bajo = np.percentile(serie, percentil_bajo)
    limite_alto = np.percentile(serie, percentil_alto)
    return limite_bajo, limite_alto

def aplicar_winsorization(X, columna, limites):
    limite_bajo, limite_alto = limites
    es_outlier = ((X[columna] < limite_bajo) | (X[columna] > limite_alto)).astype(int)
    valor_winsorizado = X[columna].clip(lower=limite_bajo, upper=limite_alto)
    return valor_winsorizado, es_outlier

def transformar_ciclica(valor, periodo):
    seno = np.sin(2 * np.pi * valor / periodo)
    coseno = np.cos(2 * np.pi * valor / periodo)
    return seno, coseno

# Winsorization
limites_windspeed = calcular_limites(X_train["windspeed"])
limites_hum = calcular_limites(X_train["hum"])

X_train["windspeed_wins"], X_train["windspeed_outlier"] = aplicar_winsorization(X_train, "windspeed", limites_windspeed)
X_test["windspeed_wins"], X_test["windspeed_outlier"] = aplicar_winsorization(X_test, "windspeed", limites_windspeed)

X_train["hum_wins"], X_train["hum_outlier"] = aplicar_winsorization(X_train, "hum", limites_hum)
X_test["hum_wins"], X_test["hum_outlier"] = aplicar_winsorization(X_test, "hum", limites_hum)

# Target Encoding
media_por_hora = y_train.groupby(X_train["hr"]).mean()
media_global = y_train.mean()

X_train["hr_target_enc"] = X_train["hr"].map(media_por_hora).fillna(media_global)
X_test["hr_target_enc"] = X_test["hr"].map(media_por_hora).fillna(media_global)

# Variables ciclicas
X_train["hr_sin"], X_train["hr_cos"] = transformar_ciclica(X_train["hr"], 24)
X_test["hr_sin"], X_test["hr_cos"] = transformar_ciclica(X_test["hr"], 24)

X_train["mnth_sin"], X_train["mnth_cos"] = transformar_ciclica(X_train["mnth"] - 1, 12)
X_test["mnth_sin"], X_test["mnth_cos"] = transformar_ciclica(X_test["mnth"] - 1, 12)

features_finales = [
    "season", "yr", "holiday", "weekday", "workingday", "weathersit",
    "temp", "atemp",
    "windspeed_wins", "windspeed_outlier",
    "hum_wins", "hum_outlier",
    "hr_target_enc",
    "hr_sin", "hr_cos",
    "mnth_sin", "mnth_cos"
]

X_train_fe = X_train[features_finales].copy()
X_test_fe = X_test[features_finales].copy()

X_train_fe.head()


,season,yr,holiday,weekday,workingday,weathersit,temp,atemp,windspeed_wins,windspeed_outlier,hum_wins,hum_outlier,hr_target_enc,hr_sin,hr_cos,mnth_sin,mnth_cos
17440,1,1,0,5,1,1,0.30,0.3030,0.1343,0,0.49,0,312.100520,-0.866025,-0.500000,-0.500000,0.866025
16867,4,1,0,2,1,1,0.48,0.4697,0.2239,0,0.67,0,309.410562,-0.965926,0.258819,-0.500000,0.866025
14851,3,1,0,1,1,1,0.62,0.6212,0.2985,0,0.38,0,309.410562,-0.965926,0.258819,-0.866025,-0.500000
5446,3,0,0,1,1,1,0.66,0.6212,0.2537,0,0.69,0,130.043706,-0.500000,0.866025,-0.500000,-0.866025
6025,3,0,0,5,1,2,0.62,0.5152,0.1642,0,0.93,1,32.613636,0.258819,0.965926,-0.866025,-0.500000


Como punto de referencia, entrenamos de nuevo el modelo de la sesion 2 con estas 17 features, sin nada nuevo todavia.


In [3]:
lr_fe = LinearRegression()
lr_fe.fit(X_train_fe, y_train)

y_pred_fe = lr_fe.predict(X_test_fe)
rmse_fe = np.sqrt(mean_squared_error(y_test, y_pred_fe))
r2_fe = r2_score(y_test, y_pred_fe)

print(f"RMSE (sesion 2, 17 features): {rmse_fe:.2f}")
print(f"R2 (sesion 2, 17 features): {r2_fe:.4f}")


RMSE (sesion 2, 17 features): 99.97
R2 (sesion 2, 17 features): 0.6799


## Parte 1: Variables de interaccion

Una **interaccion** aparece cuando el efecto de una variable sobre el target depende del valor de otra variable. Por ejemplo: es probable que la temperatura afecte la demanda de bicicletas de forma distinta en un dia laboral que en un fin de semana (en un dia laboral, la gente puede salir en bici igual aunque haga frio, para llegar al trabajo; en un fin de semana, el frio si los puede desanimar mas).

Una Regresion Lineal, por si sola, no puede capturar este tipo de efecto: solo suma el aporte de cada variable por separado. Si multiplicamos dos variables entre si, creamos una nueva columna que si le permite al modelo aprender ese efecto combinado.

Vamos a crear algunas interacciones, combinando variables que ya construimos:

- `temp_x_workingday`: temperatura combinada con dia laboral.
- `hum_x_windspeed`: humedad combinada con viento (una especie de "indice de incomodidad" climatica).
- `hrsin_x_workingday` y `hrcos_x_workingday`: la hora del dia combinada con si es un dia laboral (el patron de "hora pico" deberia ser distinto en un dia laboral que en un fin de semana).
- `season_x_temp`: la estacion del anio combinada con la temperatura.
- `weekday_x_atemp`: dia de la semana combinado con la sensacion termica. Esta la incluimos a proposito como candidata "sospechosa": no hay una razon fuerte para pensar que aporte mucho, y nos va a servir mas adelante para ver si los metodos de seleccion la detectan como poco util.


In [4]:
X_train_int = X_train_fe.copy()
X_test_int = X_test_fe.copy()

X_train_int["temp_x_workingday"] = X_train["temp"] * X_train["workingday"]
X_test_int["temp_x_workingday"] = X_test["temp"] * X_test["workingday"]

X_train_int["hum_x_windspeed"] = X_train["hum_wins"] * X_train["windspeed_wins"]
X_test_int["hum_x_windspeed"] = X_test["hum_wins"] * X_test["windspeed_wins"]

X_train_int["hrsin_x_workingday"] = X_train["hr_sin"] * X_train["workingday"]
X_test_int["hrsin_x_workingday"] = X_test["hr_sin"] * X_test["workingday"]

X_train_int["hrcos_x_workingday"] = X_train["hr_cos"] * X_train["workingday"]
X_test_int["hrcos_x_workingday"] = X_test["hr_cos"] * X_test["workingday"]

X_train_int["season_x_temp"] = X_train["season"] * X_train["temp"]
X_test_int["season_x_temp"] = X_test["season"] * X_test["temp"]

X_train_int["weekday_x_atemp"] = X_train["weekday"] * X_train["atemp"]
X_test_int["weekday_x_atemp"] = X_test["weekday"] * X_test["atemp"]

print("Features antes de las interacciones:", X_train_fe.shape[1])
print("Features despues de las interacciones:", X_train_int.shape[1])
X_train_int.head()


Features antes de las interacciones: 17
Features despues de las interacciones: 23


,season,yr,holiday,weekday,workingday,weathersit,temp,atemp,windspeed_wins,windspeed_outlier,...,hr_sin,hr_cos,mnth_sin,mnth_cos,temp_x_workingday,hum_x_windspeed,hrsin_x_workingday,hrcos_x_workingday,season_x_temp,weekday_x_atemp
17440,1,1,0,5,1,1,0.30,0.3030,0.1343,0,...,-0.866025,-0.500000,-0.500000,0.866025,0.30,0.065807,-0.866025,-0.500000,0.30,1.5150
16867,4,1,0,2,1,1,0.48,0.4697,0.2239,0,...,-0.965926,0.258819,-0.500000,0.866025,0.48,0.150013,-0.965926,0.258819,1.92,0.9394
14851,3,1,0,1,1,1,0.62,0.6212,0.2985,0,...,-0.965926,0.258819,-0.866025,-0.500000,0.62,0.113430,-0.965926,0.258819,1.86,0.6212
5446,3,0,0,1,1,1,0.66,0.6212,0.2537,0,...,-0.500000,0.866025,-0.500000,-0.866025,0.66,0.175053,-0.500000,0.866025,1.98,0.6212
6025,3,0,0,5,1,2,0.62,0.5152,0.1642,0,...,0.258819,0.965926,-0.866025,-0.500000,0.62,0.152706,0.258819,0.965926,1.86,2.5760


Entrenamos la misma Regresion Lineal, ahora con las 6 interacciones agregadas, y comparamos contra el resultado de la sesion 2.


In [5]:
lr_int = LinearRegression()
lr_int.fit(X_train_int, y_train)

y_pred_int = lr_int.predict(X_test_int)
rmse_int = np.sqrt(mean_squared_error(y_test, y_pred_int))
r2_int = r2_score(y_test, y_pred_int)

print(f"RMSE (con interacciones): {rmse_int:.2f}")
print(f"R2 (con interacciones): {r2_int:.4f}")


RMSE (con interacciones): 98.65
R2 (con interacciones): 0.6884


In [6]:
comparacion_interacciones = pd.DataFrame([
    {"modelo": "Sin interacciones (17 features)", "rmse": rmse_fe, "r2": r2_fe},
    {"modelo": "Con interacciones (23 features)", "rmse": rmse_int, "r2": r2_int},
])

comparacion_interacciones


,modelo,rmse,r2
0,Sin interacciones (17 features),99.972445,0.679914
1,Con interacciones (23 features),98.645438,0.688355


La mejora es real, aunque moderada. Esto es bastante tipico: las interacciones ayudan, pero no todas aportan lo mismo, y algunas (como `weekday_x_atemp`, que agregamos a proposito como candidata dudosa) probablemente sumen mas ruido que senial. Justo por eso necesitamos metodos de seleccion de caracteristicas: para decidir, de forma sistematica, cuales de estas 23 variables realmente vale la pena conservar.


## Parte 2: Seleccion de caracteristicas

Ya tenemos 23 variables (17 de la sesion 2 mas 6 interacciones nuevas). Vamos a aplicar las 3 familias de metodos de seleccion que vimos en la teoria, para ver cuales variables sobreviven en cada caso.


### Metodo de Filtro: Correlacion de Pearson vs Mutual Information

Los metodos de filtro comparan cada variable contra el target, una por una, sin entrenar ningun modelo. Vamos a comparar dos formas de medir esa relacion:

- **Correlacion de Pearson**: mide unicamente relaciones **lineales**. Si una variable tiene una relacion en forma de "U" o de onda con el target, Pearson puede no detectarla aunque la relacion sea fuerte.
- **Mutual Information**: mide cuanta informacion aporta una variable sobre el target, sin asumir que la relacion tiene que ser lineal. Por eso puede detectar patrones que Pearson se pierde.


In [12]:
# Calcula la correlación de Pearson (en valor absoluto) entre cada variable y el target
correlacion = X_train_int.corrwith(y_train).abs().sort_values(ascending=False)

# Calcula la Mutual Information entre cada variable y el target
# (captura tanto relaciones lineales como no lineales)
mi_scores = mutual_info_regression(
    X_train_int,
    y_train,
    random_state=RANDOM_STATE
)

# Convierte los resultados en una Serie de pandas para facilitar su lectura
mutual_info = pd.Series(
    mi_scores,
    index=X_train_int.columns
).sort_values(ascending=False)

# Reúne ambos criterios de selección en una misma tabla
ranking = pd.DataFrame({
    "correlacion_pearson": correlacion,
    "mutual_information": mutual_info
})

# Ordena las variables según su Mutual Information
ranking.sort_values("mutual_information", ascending=False)

,correlacion_pearson,mutual_information
hr_target_enc,0.707761,0.618281
hrsin_x_workingday,0.322025,0.361760
hrcos_x_workingday,0.296488,0.344305
hr_cos,0.409735,0.341014
hr_sin,0.408082,0.324893
season_x_temp,0.355566,0.163785
atemp,0.402723,0.154173
temp,0.406154,0.141386
temp_x_workingday,0.195247,0.098049
hum_wins,0.327151,0.095679


Miren con atencion las variables `hrsin_x_workingday` y `hrcos_x_workingday`: con Mutual Information aparecen entre las mas importantes, pero con Pearson quedan bastante mas abajo en el ranking. Esto tiene sentido: son variables ciclicas combinadas con una interaccion, con una relacion bastante mas compleja que una simple linea recta, y Pearson no esta capturando bien esa relacion.

Probemos ahora entrenar un modelo usando solamente las 8 variables mejor rankeadas por Mutual Information.


In [15]:
# Selecciona las 8 variables con mayor Mutual Information
top_8_mi = mutual_info.head(8).index.tolist()
print("Variables seleccionadas (Filtro - Mutual Information):", top_8_mi)

# Entrena una regresión lineal utilizando únicamente las variables seleccionadas
lr_filtro = LinearRegression()
lr_filtro.fit(X_train_int[top_8_mi], y_train)

# Genera predicciones sobre el train set
y_pred_filtro = lr_filtro.predict(X_test_int[top_8_mi])

# Calcula las métricas
rmse_filtro = np.sqrt(mean_squared_error(y_test, y_pred_filtro))
r2_filtro = r2_score(y_test, y_pred_filtro)

print(f"RMSE (Filtro, 8 features): {rmse_filtro:.2f}")
print(f"R2 (Filtro, 8 features): {r2_filtro:.4f}")

RMSE (Filtro, 8 features): 110.90
R2 (Filtro, 8 features): 0.6061


### Una aclaracion sobre los metodos de filtro

Los metodos de filtro son rapidos y simples, pero tienen una limitacion importante: evaluan cada variable **por separado**, sin considerar como interactuan entre si dentro del modelo. Por eso pueden dejar afuera variables que, combinadas con otras, si aportarian valor, o quedarse con variables redundantes entre si. Vamos a ver que los metodos Wrapper y Embedded, que si entrenan el modelo completo, suelen encontrar mejores combinaciones.

### Metodo Wrapper: Recursive Feature Elimination (RFE)

RFE entrena el modelo, elimina la variable menos importante (segun el valor absoluto de su coeficiente), y repite el proceso hasta quedarse con el numero de variables que le pidamos.

Antes de usar RFE con una Regresion Lineal, hay un detalle clave: **el valor del coeficiente depende de la escala de la variable**. Una variable medida en una escala mas chica (como `hr_sin`, entre -1 y 1) puede tener un coeficiente mas grande que una variable en otra escala (como `season`, entre 1 y 4), sin que eso signifique que sea mas importante. Por eso, antes de aplicar RFE, estandarizamos todas las variables (media 0, desvio estandar 1) con `StandardScaler`.


In [18]:
# Estandariza las variables para que todas tengan media 0 y desviación estándar 1
# (recomendado para métodos basados en regresión lineal)
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_int),
    columns=X_train_int.columns,
    index=X_train_int.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_int),
    columns=X_test_int.columns,
    index=X_test_int.index
)

# Configura RFE para seleccionar las 8 variables más relevantes
# utilizando una regresión lineal como modelo base
rfe = RFE(
    LinearRegression(),
    n_features_to_select=8
)

# Ejecuta la selección recursiva de variables
rfe.fit(X_train_scaled, y_train)

# Recupera los nombres de las variables seleccionadas
variables_rfe = X_train_scaled.columns[rfe.support_].tolist()

print("Variables seleccionadas (Wrapper - RFE):", variables_rfe)

Variables seleccionadas (Wrapper - RFE): ['season', 'yr', 'workingday', 'weathersit', 'temp', 'atemp', 'hr_target_enc', 'temp_x_workingday']


In [19]:
lr_rfe = LinearRegression()
lr_rfe.fit(X_train_scaled[variables_rfe], y_train)

y_pred_rfe = lr_rfe.predict(X_test_scaled[variables_rfe])
rmse_rfe = np.sqrt(mean_squared_error(y_test, y_pred_rfe))
r2_rfe = r2_score(y_test, y_pred_rfe)

print(f"RMSE (Wrapper - RFE, 8 features): {rmse_rfe:.2f}")
print(f"R2 (Wrapper - RFE, 8 features): {r2_rfe:.4f}")


RMSE (Wrapper - RFE, 8 features): 100.26
R2 (Wrapper - RFE, 8 features): 0.6781


### Metodo Embedded: Lasso

Lasso es una Regresion Lineal con una penalizacion adicional (regularizacion L1) que fuerza a que los coeficientes de las variables menos utiles se vuelvan **exactamente cero**. A diferencia de RFE, esto pasa en un solo entrenamiento: no hace falta ir eliminando variables una por una.

El parametro `alpha` controla que tan fuerte es esa penalizacion: con un `alpha` mas alto, mas coeficientes se vuelven cero (el modelo queda mas simple, aunque puede perder algo de precision). Igual que con RFE, usamos las variables estandarizadas, porque la penalizacion de Lasso tambien depende de la escala de cada variable.


In [20]:
# Configura el modelo Lasso
# alpha controla la intensidad de la regularización:
# valores más altos eliminan más variables (coeficientes = 0)
lasso = Lasso(
    alpha=8,
    random_state=RANDOM_STATE,
    max_iter=10000
)

# Entrena el modelo sobre las variables estandarizadas
lasso.fit(X_train_scaled, y_train)

# Recupera los coeficientes aprendidos por Lasso y los ordena
# según su magnitud (valor absoluto), de mayor a menor
coeficientes = pd.Series(
    lasso.coef_,
    index=X_train_scaled.columns
).sort_values(key=abs, ascending=False)

# Muestra los coeficientes de cada variable
coeficientes

,0
hr_target_enc,113.008042
yr,36.408429
atemp,27.992852
season_x_temp,19.191593
weathersit,-11.087864
hum_wins,-4.516065
season,1.830889
weekday_x_atemp,0.393966
mnth_cos,-0.208590
weekday,0.000000


In [21]:
# Identifica las variables cuyos coeficientes no fueron reducidos a cero por Lasso
variables_lasso = coeficientes[coeficientes != 0].index.tolist()

print("Variables que Lasso conservo (coeficiente distinto de cero):", variables_lasso)
print("Cantidad de variables conservadas:", len(variables_lasso), "de", X_train_scaled.shape[1])

# Genera predicciones sobre el conjunto de prueba
y_pred_lasso = lasso.predict(X_test_scaled)

# Calcula las métricas de desempeño del modelo
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
r2_lasso = r2_score(y_test, y_pred_lasso)

# Muestra el rendimiento del modelo Lasso
print(f"RMSE (Embedded - Lasso): {rmse_lasso:.2f}")
print(f"R2 (Embedded - Lasso): {r2_lasso:.4f}")

Variables que Lasso conservo (coeficiente distinto de cero): ['hr_target_enc', 'yr', 'atemp', 'season_x_temp', 'weathersit', 'hum_wins', 'season', 'weekday_x_atemp', 'mnth_cos']
Cantidad de variables conservadas: 9 de 23
RMSE (Embedded - Lasso): 101.70
R2 (Embedded - Lasso): 0.6688


Noten que `weekday_x_atemp`, la interaccion que agregamos a proposito como candidata dudosa, probablemente haya quedado con coeficiente cero o muy chico. Esto confirma que Lasso esta identificando correctamente cuales variables no aportan demasiado.


## Comparando los 4 enfoques

Armemos una tabla con los 4 escenarios que probamos: todas las variables, y las 3 familias de metodos de seleccion, cada uno quedandose con un subconjunto mas chico de variables.


In [ ]:
tabla_final = pd.DataFrame([
    {"enfoque": "Todas las variables (23)", "n_variables": X_train_int.shape[1], "rmse": rmse_int, "r2": r2_int},
    {"enfoque": "Filtro - Mutual Information (8)", "n_variables": len(top_8_mi), "rmse": rmse_filtro, "r2": r2_filtro},
    {"enfoque": "Wrapper - RFE (8)", "n_variables": len(variables_rfe), "rmse": rmse_rfe, "r2": r2_rfe},
    {"enfoque": "Embedded - Lasso", "n_variables": len(variables_lasso), "rmse": rmse_lasso, "r2": r2_lasso},
])

tabla_final


,enfoque,n_variables,rmse,r2
0,Todas las variables (23),23,98.645438,0.688355
1,Filtro - Mutual Information (8),8,110.898946,0.606123
2,Wrapper - RFE (8),8,100.259564,0.678073
3,Embedded - Lasso,9,101.699657,0.668758


## Cierre

Algunas conclusiones para llevarse de este material:

- **Las interacciones ayudan, pero hay que revisarlas una por una.** No todas las combinaciones de variables aportan lo mismo, y agregar interacciones sin criterio puede sumar mas ruido que valor.
- **Filtro, Wrapper y Embedded no siempre coinciden en que variables son "las mejores".** Los metodos de Filtro son los mas rapidos, pero al evaluar cada variable por separado, pueden perderse combinaciones que si funcionan bien juntas. Wrapper y Embedded, al entrenar el modelo completo, suelen encontrar subconjuntos mas competitivos.
- **Reducir variables no siempre cuesta precision.** En varios de estos casos, quedarnos con 8 o 9 variables (en vez de 23) perdimos muy poco RMSE. Un modelo mas simple es mas facil de interpretar, mas rapido de entrenar, y menos propenso a sobreajustar.
- **La escala de las variables importa.** Antes de usar metodos basados en coeficientes (RFE con Regresion Lineal, o Lasso), hay que estandarizar las variables. Si no lo hacemos, el "peso" de cada coeficiente puede reflejar la escala de la variable en lugar de su verdadera importancia.

Esta misma logica (crear variables, y despues decidir sistematicamente cuales conservar) es la que van a poder aplicar en su ejercicio final, con su propio dataset.
